# Azure AI Agent Service Diagnostic Demo


### Import Necessary Libraries
In this cell, we import all the libraries and modules required for the project.
This includes Azure AI SDKs, Gradio for UI, and custom functions.

In [1]:
import os
import json
import base64
from typing import Any, List, Dict
from dotenv import load_dotenv

# (Optional) Gradio app for UI
import gradio as gr
from gradio import ChatMessage

# Azure AI Foundry (Projects v2) — agents use azure-ai-projects only.
# `azure-ai-agents` is no longer used in this notebook.
from azure.ai.projects import AIProjectClient
import azure.ai.projects as projectslib
from azure.ai.projects.models import (
    PromptAgentDefinition,
    FileSearchTool,
    FunctionTool,
    CodeInterpreterTool,
    AutoCodeInterpreterToolParam,
    MCPTool,
)

# OpenAI types used to feed function-call results back to the agent
from openai.types.responses.response_input_param import (
    FunctionCallOutput,
    ResponseInputParam,
)

# Your custom Python functions (for "fetch_datetime", etc.)
from utils.enterprise_functions import fetch_datetime

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print("please login first")

Environment and authentication OK


### Create Client and Load Azure AI Foundry
Here, we initialize the Azure AI client using DefaultAzureCredential.
This allows us to authenticate and connect to the Azure AI service.

In [2]:
# AI Foundry Project endpoint (new Foundry Projects v2 API).
# Format: https://<account>.services.ai.azure.com/api/projects/<project>
project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
)
openai_client = project_client.get_openai_client()

print("project_client api version:", project_client._config.api_version)
print(f"azure-ai-projects version: {projectslib.__version__}")

project_client api version: v1
azure-ai-projects version: 2.1.0


### Set Up Tools (BingGroundingTool, FileSearchTool)
In this step, we configure tools such as `BingGroundingTool` and `FileSearchTool`.
We check for existing connections and create or reuse vector stores for document search.

Note:
If you see the following cell has error:
```
AzureCliCredential: Please run 'az login' to set up an account
```

relogin from powershell
```powershell
az logout
az account clear
az login --tenant 00000000-0000-0000-0000-000000000000
```


In [3]:
# --- File Search tool: vector store lives on the OpenAI client ---
FOLDER_NAME = os.path.join("..", "treatment-data")
VECTOR_STORE_NAME = "lung-treatment-vector-store"

all_vector_stores = list(openai_client.vector_stores.list())
existing_vector_store = next(
    (store for store in all_vector_stores if store.name == VECTOR_STORE_NAME),
    None,
)

vector_store_id = None
if existing_vector_store:
    vector_store_id = existing_vector_store.id
    print(f"reusing vector store > {existing_vector_store.name} (id: {existing_vector_store.id})")
else:
    if os.path.isdir(FOLDER_NAME):
        vector_store = openai_client.vector_stores.create(name=VECTOR_STORE_NAME)
        vector_store_id = vector_store.id
        print(f"created vector store > {vector_store.name} (id: {vector_store_id})")

        for file_name in os.listdir(FOLDER_NAME):
            file_path = os.path.join(FOLDER_NAME, file_name)
            if os.path.isfile(file_path):
                print(f"uploading > {file_name}")
                with open(file_path, "rb") as f:
                    openai_client.vector_stores.files.upload_and_poll(
                        vector_store_id=vector_store_id,
                        file=f,
                    )

file_search_tool = None
if vector_store_id:
    file_search_tool = FileSearchTool(vector_store_ids=[vector_store_id])

reusing vector store > lung-treatment-vector-store (id: vs_wa0AUVQAMare9GHkyjoTwC55)


### Set Up Built-in Code Interpreter
Upload CSV data files via `openai_client.files.create()` and attach them to a
`CodeInterpreterTool` with `AutoCodeInterpreterToolParam`.

In [4]:
# --- Built-in Code Interpreter (upload CSV files via openai_client) ---
DATA_FOLDER_NAME = os.path.join("..", "medicare-data")
# Set to True to force re-upload (e.g. after changing CSV schema/data).
FORCE_REUPLOAD = False # True

# List existing files uploaded to the project
existing_files = list(openai_client.files.list())
existing_files_dict = {f.filename: f.id for f in existing_files}

data_file_ids = []
if os.path.isdir(DATA_FOLDER_NAME):
    for file_name in sorted(os.listdir(DATA_FOLDER_NAME)):
        # Only upload CSV files (skip metadata.md, etc.)
        if not file_name.lower().endswith(".csv"):
            continue
        file_path = os.path.join(DATA_FOLDER_NAME, file_name)
        if not os.path.isfile(file_path):
            continue

        if file_name in existing_files_dict and not FORCE_REUPLOAD:
            data_file_id = existing_files_dict[file_name]
            data_file_ids.append(data_file_id)
            print(f"file already uploaded > {file_name} (id: {data_file_id})")
        else:
            # Delete stale version if it exists
            if file_name in existing_files_dict:
                old_id = existing_files_dict[file_name]
                openai_client.files.delete(old_id)
                print(f"deleted stale file > {file_name} (id: {old_id})")
            print(f"uploading > {file_name}")
            with open(file_path, "rb") as f:
                uploaded = openai_client.files.create(purpose="assistants", file=f)
            data_file_ids.append(uploaded.id)
            print(f"uploaded  > {file_name} (id: {uploaded.id})")

code_interpreter_tool = None
if data_file_ids:
    code_interpreter_tool = CodeInterpreterTool(
        container=AutoCodeInterpreterToolParam(file_ids=data_file_ids)
    )
    print(f"code interpreter > {len(data_file_ids)} files attached")
else:
    print("no data files found to upload")

file already uploaded > lung_cancer_patient_clinical_features.csv (id: assistant-56r1u6vfZp2KaqhqYedKVJ)
file already uploaded > lung_cancer_patients_symptoms_2025.csv (id: assistant-VCEApMVN8g497V1i28dzh8)
code interpreter > 2 files attached


### (Optional) Custom Code Interpreter via MCPTool

A custom code interpreter gives you **full control** over the runtime environment
(custom Python packages, container images, compute resources) using Azure Container
Apps Dynamic Sessions. It exposes a Model Context Protocol (MCP) server.

**Prerequisites (one-time infrastructure setup):**
1. Register the preview feature: `az feature register --namespace Microsoft.App --name SessionPoolsSupportMCP`
2. Deploy `infra/custom-code-interpreter.bicep` to your resource group (see next cell)
3. Set `AZURE_AI_CONNECTION_ID` in your `.env` file from the deployment output

> **Note:** The built-in `CodeInterpreterTool` above works out of the box. Only
> use the custom code interpreter when you need specific Python packages or
> dedicated compute. See [docs](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/custom-code-interpreter?pivots=python).

In [5]:
# --- (Optional) Deploy custom code interpreter infrastructure ---
# Run this cell ONCE to write the Bicep template to disk.
# Then deploy manually with the az CLI commands in the next markdown cell.

BICEP_DIR = "infra"
BICEP_FILE = os.path.join(BICEP_DIR, "custom-code-interpreter.bicep")
os.makedirs(BICEP_DIR, exist_ok=True)

bicep_content = r'''
@description('Suffix for resource names to ensure uniqueness')
@minLength(3)
param suffix string = uniqueString(resourceGroup().id)

@description('Container Apps environment name')
@minLength(3)
param environmentName string = 'aca-env-${suffix}'

@description('Session pool name')
@minLength(3)
param sessionPoolName string = 'sp-${suffix}'

@description('CPU per container instance (vCPU)')
@minValue(1)
@maxValue(16)
param cpu int = 1

@description('RAM per container instance (GiB)')
@minValue(1)
@maxValue(16)
param memory int = 2

@description('Location of all ACA resources.')
@allowed([
  'eastus'
  'swedencentral'
  'northeurope'
])
param location string = 'swedencentral'

@description('Use managed identity for deployment script principal')
param useManagedIdentity bool = true

@description('Code interpreter container image')
param image string = 'mcr.microsoft.com/k8se/services/codeinterpreter:0.9.18-python3.12'

// --- Container Apps Environment ---
resource environment 'Microsoft.App/managedEnvironments@2025-10-02-preview' = {
  name: environmentName
  location: location
  properties: {
    workloadProfiles: [
      {
        name: 'Consumption'
        workloadProfileType: 'Consumption'
      }
    ]
  }
}

// --- Dynamic Session Pool ---
resource sessionPool 'Microsoft.App/sessionPools@2025-10-02-preview' = {
  name: sessionPoolName
  location: location
  properties: {
    environmentId: environment.id
    poolManagementType: 'Dynamic'
    containerType: 'CustomContainer'
    scaleConfiguration: {
      maxConcurrentSessions: 10
      readySessionInstances: 5
    }
    dynamicPoolConfiguration: {
      lifecycleConfiguration: {
        cooldownPeriodInSeconds: 600
        lifecycleType: 'Timed'
      }
    }
    customContainerTemplate: {
      containers: [
        {
          name: 'jupyterpython'
          image: image
          env: [
            { name: 'SYS_RUNTIME_SANDBOX', value: 'AzureContainerApps-DynamicSessions' }
            { name: 'AZURE_CODE_EXEC_ENV', value: 'AzureContainerApps-DynamicSessions-Py3.12' }
            { name: 'AZURECONTAINERAPPS_SESSIONS_SANDBOX_VERSION', value: '7758' }
            { name: 'JUPYTER_TOKEN', value: 'AzureContainerApps-DynamicSessions' }
          ]
          resources: {
            cpu: cpu
            memory: '${memory}Gi'
          }
          probes: [
            { type: 'Liveness', httpGet: { path: '/health', port: 6000 }, failureThreshold: 4 }
            { type: 'Startup', httpGet: { path: '/health', port: 6000 }, failureThreshold: 30, periodSeconds: 2 }
          ]
        }
      ]
      ingress: { targetPort: 6000 }
    }
    mcpServerSettings: { isMcpServerEnabled: true }
    sessionNetworkConfiguration: { status: 'egressEnabled' }
  }
}

// --- Managed Identity for deploy script ---
resource scriptPrincipal 'Microsoft.ManagedIdentity/userAssignedIdentities@2023-01-31' = if (useManagedIdentity) {
  name: 'deployScriptIdentity-${suffix}'
  location: location
}

resource roleAssignment 'Microsoft.Authorization/roleAssignments@2022-04-01' = if (useManagedIdentity) {
  name: guid(scriptPrincipal!.id, 'apps-sessionpool-contributor')
  scope: resourceGroup()
  properties: {
    roleDefinitionId: subscriptionResourceId(
      'Microsoft.Authorization/roleDefinitions',
      'f7669afb-68b2-44b4-9c5f-6d2a47fddda0' // Container Apps SessionPools Contributor
    )
    principalId: scriptPrincipal!.properties.principalId
    principalType: 'ServicePrincipal'
  }
}

// --- Deployment script to fetch MCP key ---
resource deployScript 'Microsoft.Resources/deploymentScripts@2020-10-01' = {
  name: 'getmcpkey-${suffix}'
  location: location
  kind: 'AzureCLI'
  identity: useManagedIdentity ? {
    type: 'UserAssigned'
    userAssignedIdentities: { '${scriptPrincipal!.id}': {} }
  } : null
  properties: {
    azCliVersion: '2.77.0'
    scriptContent: 'az rest --method post --url "$SESSION_POOL_ID/fetchMCPServerCredentials?api-version=2025-02-02-preview" | jq -c \'{"key": .apiKey}\' > $AZ_SCRIPTS_OUTPUT_PATH'
    timeout: 'PT30M'
    retentionInterval: 'P1D'
    cleanupPreference: 'OnSuccess'
    environmentVariables: [ { name: 'SESSION_POOL_ID', value: sessionPool.id } ]
  }
}

// --- AI Account + Project + Connection ---
// NOTE: If you already have a Foundry project, remove this resource block
// and create the connection manually or via a separate Bicep module.
resource aiAccount 'Microsoft.CognitiveServices/accounts@2025-10-01-preview' = {
  name: 'aia-${suffix}'
  location: location
  kind: 'AIServices'
  sku: { name: 'S0' }
  properties: {
    customSubDomainName: 'myaiaccount-${suffix}'
    allowProjectManagement: true
  }

  resource project 'projects' = {
    name: 'aip-${suffix}s'
    properties: { description: 'Custom code interpreter project.' }

    resource mcpConn 'connections' = {
      name: 'aic-${suffix}'
      properties: {
        authType: 'CustomKeys'
        category: 'RemoteTool'
        credentials: { keys: { 'x-ms-apikey': deployScript.properties.outputs.key } }
        target: sessionPool.properties.mcpServerSettings.mcpServerEndpoint
      }
    }
  }
}

@description('Project connection ID for the Code Interpreter MCP Tool')
output AZURE_AI_CONNECTION_ID string = aiAccount::project::mcpConn.id

@description('AI Project Endpoint')
output AZURE_AI_PROJECT_ENDPOINT string = aiAccount::project.properties.endpoints['AI Foundry API']
'''.strip()

with open(BICEP_FILE, "w") as f:
    f.write(bicep_content)

print(f"wrote > {BICEP_FILE}")
print("deploy with: az deployment group create --template-file ./infra/custom-code-interpreter.bicep ...")

wrote > infra/custom-code-interpreter.bicep
deploy with: az deployment group create --template-file ./infra/custom-code-interpreter.bicep ...


### Deploy / Tear Down Custom Code Interpreter Infrastructure

**Register the preview feature (one-time):**
```bash
az feature register --namespace Microsoft.App --name SessionPoolsSupportMCP
az provider register -n Microsoft.App
```

**Deploy** (replace placeholders with your values):
```bash
az deployment group create \
    --name custom-code-interpreter \
    --subscription <YOUR_SUBSCRIPTION> \
    --resource-group <YOUR_RESOURCE_GROUP> \
    --template-file ./infra/custom-code-interpreter.bicep \
    --parameters location=swedencentral
```
> ⏳ Deployment can take **up to 1 hour** (session pool allocation is slow).

**Tear down** (delete the entire resource group or just the deployment):
```bash
# Option 1: Delete just the deployment resources
az deployment group delete \
    --name custom-code-interpreter \
    --subscription <YOUR_SUBSCRIPTION> \
    --resource-group <YOUR_RESOURCE_GROUP>

# Option 2: Delete the whole resource group (if dedicated)
az group delete --name <YOUR_RESOURCE_GROUP> --yes --no-wait
```

After deployment, copy `AZURE_AI_CONNECTION_ID` from the output into your `.env` file.

### Combine All Tools into a List
In Projects v2, agents take a plain `tools=[...]` list — there's no `ToolSet`
and no `enable_auto_function_calls`. Function-call execution happens explicitly
in the Responses loop (see the chat handler below).

In [6]:
# --- Declarative function tool: fetch_datetime ---
# In v2, function tools are declared by JSON Schema. The model emits
# `function_call` items in the response stream; we execute them and feed the
# result back via `FunctionCallOutput`.
fetch_datetime_tool = FunctionTool(
    name="fetch_datetime",
    description="Returns the current UTC date/time as JSON.",
    parameters={
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
    strict=True,
)

# Map function name -> Python callable so the chat loop can dispatch.
function_dispatch: Dict[str, Any] = {
    "fetch_datetime": lambda **_: fetch_datetime(),
}

# --- Build the tools list in the order the agent should consider them ---
tools: List[Any] = []
if file_search_tool:
    tools.append(file_search_tool)
if code_interpreter_tool:
    tools.append(code_interpreter_tool)
tools.append(fetch_datetime_tool)

# --- (Optional) Custom Code Interpreter via MCPTool ---
# Uncomment the block below AFTER deploying infra/custom-code-interpreter.bicep
# and setting AZURE_AI_CONNECTION_ID in your .env file.
# This REPLACES the built-in CodeInterpreterTool above.
#
# custom_code_interpreter = MCPTool(
#     server_url="https://localhost",  # placeholder; real URL is in the connection
#     server_label="python_tool",
#     require_approval="never",
#     allowed_tools=["launchShell", "runPythonCodeInRemoteEnvironment"],
#     project_connection_id=os.environ.get("AZURE_AI_CONNECTION_ID", ""),
# )
# tools.append(custom_code_interpreter)

for t in tools:
    print(f"tool > {type(t).__name__} (type={getattr(t, 'type', '?')})")

tool > FileSearchTool (type=file_search)
tool > CodeInterpreterTool (type=code_interpreter)
tool > FunctionTool (type=function)


### Create or Update the Medical Diagnostic Agent (versioned)
In Projects v2, agents are versioned resources. Each call to `create_version`
either creates the agent (if it doesn't exist) or pins a new version of an
existing agent. There is no in-place `update_agent` — just create a new version
with the desired model, instructions, and tools.

In [7]:
AGENT_NAME = "med-diagnostic-agent"
model_name = settings.model_deployment_name

# Chart style note: STEP 1 gives principles (LOF + feature importance + scatter)
# but lets the code interpreter decide implementation details like markers/colors.
# Edit the STEP 1 text below to change the analysis or chart approach.
instructions = (
    "You are a helpful medical diagnostic assistant.\n\n"
    "## Tools:\n"
    " * code_interpreter: analyze patient data from CSV files. "
    "You have two CSV datasets:\n"
    "   - lung_cancer_patient_clinical_features.csv: PATIENT_ID, GENDER, AGE, risk factors "
    "(SMOKING, YELLOW_FINGERS, ANXIETY, PEER_PRESSURE, CHRONIC_DISEASE, FATIGUE, ALLERGY, "
    "WHEEZING, ALCOHOL_CONSUMING, COUGHING, SHORTNESS_OF_BREATH, SWALLOWING_DIFFICULTY, "
    "CHEST_PAIN), and LUNG_CANCER diagnosis (YES/NO).\n"
    "   - lung_cancer_patients_symptoms_2025.csv: PATIENT_ID, CANCER_STAGE, TUMOR_SIZE_CM, "
    "DIAGNOSIS_DATE, TREATMENT_RECEIVED.\n"
    "   Always load BOTH CSV files and join them on PATIENT_ID (inner join) to get the "
    "complete patient profile before analysis.\n"
    " * file_search: search treatment best practices knowledge base for stage-specific "
    "protocols, drug regimens, and risk-factor-based recommendations.\n"
    " * fetch_datetime: get the current date and time.\n"
    "\n"
    "## CRITICAL: Multi-Tool Workflow (MUST follow this order):\n"
    "When asked to analyze, examine, assess, or look at a patient — even if the user \n"
    "does NOT explicitly ask for a chart or visualization — ALWAYS generate a scatter \n"
    "plot as part of the analysis. The chart is a mandatory deliverable.\n"
    "STEP 1 (code_interpreter FIRST): Load both CSVs, join on PATIENT_ID, extract the "
    "patient's full profile. Then:\n"
    "  a) Normalize all numeric features and run LOF outlier detection.\n"
    "  b) Identify the 2 most discriminative features (use feature importance, variance, "
    "or PCA loadings — your choice). Exclude binary/categorical columns (Yes/No, 0/1) "
    "from axis selection — prefer continuous features like AGE, TUMOR_SIZE_CM, or PCA "
    "components for better scatter spread.\n"
    "  c) Generate a clean 2D scatter plot of those 2 features: all patients as a "
    "point cloud, the TARGET patient clearly highlighted (different color/shape/size). "
    "Only highlight the target patient, not other outliers. Include a legend and title.\n"
    "  d) Summarize the patient's full profile: age, gender, cancer stage, tumor size, "
    "risk factors, LOF score, outlier status, and which features were most informative.\n"
    "STEP 2 (file_search AFTER code_interpreter completes): Search best practices for "
    "the patient's CANCER_STAGE (e.g., 'Stage II NSCLC treatment') and relevant risk "
    "factors found in Step 1 (e.g., 'smoking patient treatment', 'young patient lung cancer').\n"
    "STEP 3 (synthesize): Combine the data analysis from Step 1 with the best practice "
    "guidelines from Step 2 into a personalized treatment recommendation with specific "
    "drug regimens, citing the patient's stage, risk factors, and guideline sources.\n"
    "\n"
    "IMPORTANT: Always run code_interpreter BEFORE file_search. Never skip the data "
    "analysis step. The file_search query must be informed by the patient's actual "
    "cancer stage and risk factors discovered in Step 1.\n"
    "\n"
    "## Chart / Plot generation:\n"
    "ALWAYS generate a 2D scatter plot when analyzing a patient, even if the user "
    "does not explicitly request one. The chart is a core part of every patient analysis.\n"
    "When generating charts or plots with matplotlib, ALWAYS use the non-interactive Agg "
    "backend and save figures to a file instead of calling plt.show(). Use this pattern:\n"
    "```python\n"
    "import matplotlib\n"
    "matplotlib.use('Agg')\n"
    "import matplotlib.pyplot as plt\n"
    "# ... create your plot ...\n"
    "plt.savefig('output.png', dpi=150, bbox_inches='tight')\n"
    "plt.close()\n"
    "```\n"
    "Never call plt.show(). Always save to a .png file and reference it in your response.\n"
    "\n"
    "## Data handling:\n"
    "When encountering missing or NaN values in the data, handle them automatically "
    "without asking the user for confirmation. Use this approach:\n"
    "- For numeric columns: fill missing values with the column median.\n"
    "- For categorical columns: fill missing values with the column mode.\n"
    "- Never drop rows unless fewer than 5% are affected.\n"
    "- Never ask the user how to handle missing data. Just proceed.\n"
    "\n"
    "## Guidelines:\n"
    "Provide well-structured and professional answers.\n"
    "When recommending treatments, always cite the cancer stage, relevant risk factors, "
    "and specific drug names or regimens from the best practices knowledge base.\n"
)

agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=instructions,
        tools=tools,
    ),
    description="Medical diagnostic agent with File Search, Code Interpreter, and datetime function.",
)
print(f"agent > {agent.name} (id: {agent.id}, version: {agent.version})")

agent > med-diagnostic-agent (id: med-diagnostic-agent:15, version: 15)


### Create a Conversation
In Projects v2, multi-turn state is held by an OpenAI **conversation**
(via `openai_client.conversations.create()`). It replaces the old `thread`
concept and is what `responses.create(conversation=...)` reads from / writes to.

In [8]:
conversation = openai_client.conversations.create()
print(f"conversation > created (id: {conversation.id})")

conversation > created (id: conv_4c6746ebddc059fd00XfFnlZct4Bdg6GedEwq75S1oLNvpB6hj)


### Streaming Helpers
Projects v2 streams an OpenAI **Responses** event sequence (no `AgentEventHandler`
subclass). The helper functions below handle tool-call bubbles and assistant text
in the Gradio chat. When the model emits a `function_call` item, we execute the
matching Python callable, send a `FunctionCallOutput` back via a follow-up
`responses.create`, and continue streaming.

In [9]:
import re

# Set to True to print event-level logs to the notebook cell output.
DEBUG = True

# ---------------------------------------------------------------------------
# Image-ref stripping: WHITELIST approach (not blacklist)
# ---------------------------------------------------------------------------
# Problem: The code-interpreter agent writes image refs in many unpredictable
# formats — sandbox:/mnt/data/..., bare filenames (plot.png), /mnt/data/...,
# and <img> HTML tags. None of these resolve in the Gradio browser context.
#
# The ONLY valid images are base64 data: URIs that we inject after downloading
# from the containers API. Two paths produce these:
#   1. container_file_citation annotations (handled in _format_annotation)
#   2. Container-file fallback (handled in _stream_response message-done)
#
# Previous blacklist approach (separate regexes for sandbox, local, html) kept
# missing new patterns. Whitelist: strip ALL image markdown/html UNLESS the URL
# starts with "data:". Two regexes cover markdown-style and HTML-style.
# ---------------------------------------------------------------------------
_BROKEN_IMG_RE = re.compile(r'!?\[[^\]]*\]\((?!data:)[^)]*\)')
_BROKEN_HTML_RE = re.compile(r'<img[^>]*(?!data:)[^>]*>', re.IGNORECASE)

# UI labels for tool-call bubbles in the Gradio chat.
FUNCTION_TITLES = {
    "fetch_datetime": "🕒 fetching datetime",
    "file_search": "📄 searching docs",
    "code_interpreter": "📊 analyzing data",
}

# Map a streamed output-item type to a function-title key.
TOOL_ITEM_TO_KEY = {
    "file_search_call": "file_search",
    "code_interpreter_call": "code_interpreter",
}

AGENT_REFERENCE = {"name": AGENT_NAME, "type": "agent_reference"}


def _tool_title(name: str) -> str:
    return FUNCTION_TITLES.get(name, f"🛠 calling {name}")


# ---------------------------------------------------------------------------
# Image download helpers
# ---------------------------------------------------------------------------
def _download_image_b64(container_id: str, file_id: str, filename: str) -> str | None:
    """Download a file from a code-interpreter container and return as base64 data URI.

    Returns None on failure. Used by:
      - _format_annotation (for container_file_citation annotations)
      - _fetch_container_images (fallback when annotations are missing)
    """
    try:
        content = openai_client.containers.files.content.retrieve(
            file_id=file_id, container_id=container_id
        )
        image_bytes = content.read()
        if DEBUG:
            print(f"  image downloaded > {len(image_bytes)} bytes (container={container_id}, file={file_id})")
        b64 = base64.b64encode(image_bytes).decode("utf-8")
        ext = filename.rsplit(".", 1)[-1].lower()
        mime = {
            "png": "image/png", "jpg": "image/jpeg",
            "jpeg": "image/jpeg", "gif": "image/gif",
            "svg": "image/svg+xml",
        }.get(ext, "image/png")
        return f"data:{mime};base64,{b64}"
    except Exception as e:
        if DEBUG:
            print(f"  image download FAILED > {type(e).__name__}: {e}")
        return None


def _fetch_container_images(container_id: str) -> dict[str, str]:
    """List all image files in a container and download them as base64 data URIs.

    Returns a dict mapping filename -> data:URI string.
    This is the FALLBACK for when the agent writes image refs in its text
    but doesn't emit container_file_citation annotations. We list all files
    in the container via openai_client.containers.files.list() and download
    any that look like images.
    """
    images: dict[str, str] = {}
    try:
        files = openai_client.containers.files.list(container_id=container_id)
        for f in files:
            fname = getattr(f, "filename", "") or getattr(f, "name", "") or ""
            fid = getattr(f, "id", "")
            if DEBUG:
                print(f"  container file > {fname} (id={fid})")
            if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.svg')):
                uri = _download_image_b64(container_id, fid, fname)
                if uri:
                    # Store by both full path and basename for flexible matching
                    images[fname] = uri
                    basename = fname.rsplit("/", 1)[-1] if "/" in fname else fname
                    images[basename] = uri
    except Exception as e:
        if DEBUG:
            print(f"  container file list FAILED > {type(e).__name__}: {e}")
    return images


def _format_annotation(ann) -> str:
    """Render a Responses annotation as a Markdown link or inline image.

    This is one of two places images enter the chat as valid data: URIs.
    container_file_citation annotations carry a file_id + container_id;
    we download the bytes and emit an inline ![](data:image/png;base64,...).
    The other path is _fetch_container_images (fallback in _stream_response).
    """
    a_type = getattr(ann, "type", None)
    if DEBUG:
        print(f"  annotation > type={a_type}")
    if a_type == "url_citation":
        title = getattr(ann, "title", "") or "source"
        return f" [{title}]({ann.url})"
    if a_type == "file_citation":
        filename = getattr(ann, "filename", "") or "file"
        return f" [{filename}]"
    if a_type == "container_file_citation":
        filename = getattr(ann, "filename", "") or "file"
        file_id = getattr(ann, "file_id", "")
        container_id = getattr(ann, "container_id", "")
        if DEBUG:
            print(f"  container_file_citation > filename={filename} file_id={file_id} container_id={container_id}")
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.svg')):
            uri = _download_image_b64(container_id, file_id, filename)
            if uri:
                return f"\n\n![{filename}]({uri})"
            return f" [{filename}]"
        return f" [{filename}]"
    return ""

### Implement the Main Chat Functions
These functions define how user messages and tool interactions are processed.
It uses the Responses API streaming to handle conversations and streams partial
responses, resolving function calls in a loop until the agent produces a final
message.

In [10]:
def convert_dict_to_chatmessage(msg: dict) -> ChatMessage:
    """Convert a legacy dict-based history entry to a gr.ChatMessage."""
    return ChatMessage(
        role=msg["role"],
        content=msg["content"],
        metadata=msg.get("metadata", None),
    )


def _stream_response(resp_iter, chat, in_progress_tools: Dict[str, ChatMessage]):
    """Consume one streamed Responses iterator.

    Yields `(chat, "")` after each visible state change so Gradio can
    repaint. Returns a list of pending function calls that the caller must
    execute and feed back via a follow-up `responses.create`.
    """
    pending_function_calls: List[Any] = []
    last_response_id: str | None = None
    assistant_msg: ChatMessage | None = None
    _text_buf_len = 0  # track chars since last yield for throttling
    # ---------------------------------------------------------------------------
    # Container-image fallback: collect container_ids from code_interpreter done
    # events. If the final message references images but lacks
    # container_file_citation annotations, we list container files and inject
    # the images ourselves.
    # ---------------------------------------------------------------------------
    ci_container_ids: list[str] = []

    for event in resp_iter:
        etype = getattr(event, "type", "")

        if etype == "response.created":
            last_response_id = event.response.id
            if DEBUG:
                print(f"status > created (response_id: {last_response_id})")

        elif etype == "response.output_item.added":
            item = event.item
            item_type = getattr(item, "type", "")
            if DEBUG:
                print(f"output_item > {item_type} (id: {item.id})")

            if item_type == "message":
                assistant_msg = ChatMessage(
                    role="assistant",
                    content="",
                    metadata={"id": item.id},
                )
                chat.append(assistant_msg)
                yield chat, ""

            elif item_type == "function_call":
                if DEBUG:
                    print(f"tool call > {item.name} (call_id: {item.call_id})")
                bubble = ChatMessage(
                    role="assistant",
                    content="",
                    metadata={
                        "title": _tool_title(item.name),
                        "status": "pending",
                        "id": f"tool-{item.call_id}",
                    },
                )
                chat.append(bubble)
                in_progress_tools[item.call_id] = bubble
                yield chat, ""

            elif item_type in TOOL_ITEM_TO_KEY:
                key = TOOL_ITEM_TO_KEY[item_type]
                if DEBUG:
                    print(f"tool call > {key}")
                bubble = ChatMessage(
                    role="assistant",
                    content="searching...",
                    metadata={
                        "title": _tool_title(key),
                        "status": "pending",
                        "id": f"tool-{item.id}",
                    },
                )
                chat.append(bubble)
                in_progress_tools[item.id] = bubble
                yield chat, ""

        elif etype == "response.function_call_arguments.delta":
            bubble = in_progress_tools.get(getattr(event, "call_id", ""))
            if bubble is not None:
                bubble.content = (bubble.content or "") + event.delta
                yield chat, ""

        elif etype == "response.output_item.done":
            item = event.item
            item_type = getattr(item, "type", "")
            if item_type == "function_call":
                if DEBUG:
                    print(f"tool done > {item.name} args={item.arguments}")
                pending_function_calls.append(item)
                bubble = in_progress_tools.pop(item.call_id, None)
                if bubble is not None:
                    bubble.metadata["status"] = "done"
                    bubble.content = item.arguments or ""
                yield chat, ""
            elif item_type in TOOL_ITEM_TO_KEY:
                key = TOOL_ITEM_TO_KEY[item_type]
                if DEBUG:
                    print(f"tool done > {key}")
                # Capture container_id from code_interpreter for fallback image download
                cid = getattr(item, "container_id", None)
                if cid and cid not in ci_container_ids:
                    ci_container_ids.append(cid)
                    if DEBUG:
                        print(f"  captured container_id > {cid}")
                bubble = in_progress_tools.pop(item.id, None)
                if bubble is not None:
                    bubble.metadata["status"] = "done"
                yield chat, ""
            elif item_type == "message":
                if DEBUG:
                    print(f"message done > (id: {item.id})")
                # Decorate the final message with citation links / inline images
                if assistant_msg is not None:
                    for content_block in getattr(item, "content", []) or []:
                        block_type = getattr(content_block, "type", None)
                        if DEBUG:
                            print(f"  content block > type={block_type}")
                        if block_type == "output_text":
                            # --- RAW OUTPUT LOGGING ---
                            raw_block_text = getattr(content_block, "text", "")
                            if DEBUG:
                                print(f"  === RAW BLOCK TEXT (last 500 chars) ===")
                                print(f"  {raw_block_text[-500:]}")

                            # Step 1: Process annotations (container_file_citation -> data: URI)
                            annotations = getattr(content_block, "annotations", []) or []
                            if DEBUG:
                                print(f"  annotations count > {len(annotations)}")
                            has_image_annotation = False
                            for ann in annotations:
                                a_type = getattr(ann, "type", None)
                                if a_type == "container_file_citation":
                                    has_image_annotation = True
                                assistant_msg.content += _format_annotation(ann)

                            # Step 2: Fallback — scan containers for images when:
                            #   a) Text has broken image refs but no container_file_citation, OR
                            #   b) No images were injected at all but containers exist
                            # This handles the case where the agent generates a plot in
                            # code_interpreter but doesn't emit a citation for it.
                            raw_text = assistant_msg.content
                            broken_refs = _BROKEN_IMG_RE.findall(raw_text)
                            has_broken_image_refs = any(
                                r for r in broken_refs
                                if any(r.lower().endswith(ext + ')') for ext in ['.png', '.jpg', '.jpeg', '.gif', '.svg'])
                                or any(r.lower().endswith(ext + ']') for ext in ['.png', '.jpg', '.jpeg', '.gif', '.svg'])
                            )
                            need_fallback = (
                                ci_container_ids
                                and not has_image_annotation
                                and (has_broken_image_refs or ci_container_ids)
                            )
                            if need_fallback:
                                if DEBUG:
                                    print(f"  FALLBACK: scanning {len(ci_container_ids)} containers for images...")
                                all_images: dict[str, str] = {}
                                for cid in ci_container_ids:
                                    all_images.update(_fetch_container_images(cid))
                                if all_images:
                                    # First try to replace broken refs by matching filenames
                                    if has_broken_image_refs:
                                        import re as _re
                                        def _replace_broken_ref(m):
                                            full = m.group(0)
                                            url_match = _re.search(r'\(([^)]+)\)', full)
                                            if url_match:
                                                url = url_match.group(1)
                                                fname = url.rsplit("/", 1)[-1] if "/" in url else url
                                            else:
                                                fname = full.strip("[]!")
                                            if fname in all_images:
                                                if DEBUG:
                                                    print(f"  FALLBACK: replaced {fname} with data: URI")
                                                return f"\n\n![{fname}]({all_images[fname]})"
                                            return full
                                        raw_text = _BROKEN_IMG_RE.sub(_replace_broken_ref, raw_text)
                                        assistant_msg.content = raw_text
                                    # If no image was injected yet (annotation or ref-replace),
                                    # append all found images at the end of the message
                                    if not has_image_annotation and 'data:image' not in assistant_msg.content:
                                        if DEBUG:
                                            print(f"  FALLBACK: appending {len(all_images)} container images to message")
                                        seen = set()
                                        for fname, uri in all_images.items():
                                            if uri not in seen:
                                                seen.add(uri)
                                                assistant_msg.content += f"\n\n![{fname}]({uri})"
                                                if DEBUG:
                                                    print(f"  appended > {fname}")

                            # Step 3: Strip any remaining broken refs (no match found)
                            raw_text = assistant_msg.content
                            cleaned = _BROKEN_IMG_RE.sub('', raw_text)
                            cleaned = _BROKEN_HTML_RE.sub('', cleaned)
                            if cleaned != raw_text:
                                assistant_msg.content = cleaned
                                if DEBUG:
                                    print(f"  stripped remaining broken image refs from final text")

                assistant_msg = None
                yield chat, ""

        elif etype == "response.output_text.delta":
            if assistant_msg is None:
                assistant_msg = ChatMessage(role="assistant", content="")
                chat.append(assistant_msg)
            assistant_msg.content = (assistant_msg.content or "") + event.delta
            # Live-strip broken image refs during streaming so the user never
            # sees a flash of broken markdown images before the final cleanup.
            # Uses same whitelist regexes — only data: URIs survive.
            cleaned = _BROKEN_IMG_RE.sub('', assistant_msg.content)
            cleaned = _BROKEN_HTML_RE.sub('', cleaned)
            if cleaned != assistant_msg.content:
                assistant_msg.content = cleaned
            _text_buf_len += len(event.delta)
            # Throttle UI updates: yield every ~80 chars or on newlines
            if _text_buf_len >= 80 or "\n" in event.delta:
                _text_buf_len = 0
                yield chat, ""

        elif etype == "response.completed":
            if DEBUG:
                print("status > completed")
            yield chat, ""

        elif etype == "response.failed":
            err = getattr(event.response, "error", None)
            if DEBUG:
                print(f"status > failed: {err}")
            chat.append(ChatMessage(
                role="assistant",
                content=f"error > {err}",
            ))
            yield chat, ""

    # Stash the response id on the iterator for the caller
    resp_iter._last_response_id = last_response_id  # type: ignore[attr-defined]
    resp_iter._pending_calls = pending_function_calls  # type: ignore[attr-defined]


def azure_enterprise_chat(user_message: str, history: List[dict]):
    """Stream a multi-turn agent reply through the Responses API.

    Iteratively resolves any function calls until the agent produces a final
    message. Uses the global `conversation` for multi-turn state.

    Architecture:
    - Uses openai_client.responses.create(stream=True) with conversation.id
    - agent_reference routes the request to our named agent
    - _stream_response handles events, yields Gradio chat snapshots
    - When the agent calls tools (function_call items), we execute them
      locally and feed results back via a follow-up responses.create()
    - This loop repeats until no more pending function calls remain
    """
    # Convert existing history from dict to ChatMessage
    chat: List[ChatMessage] = [convert_dict_to_chatmessage(m) for m in history]
    chat.append(ChatMessage(role="user", content=user_message))
    yield chat, ""

    in_progress_tools: Dict[str, ChatMessage] = {}

    # First request: send the user prompt
    if DEBUG:
        print(f"\nuser > {user_message}")
    stream = openai_client.responses.create(
        stream=True,
        conversation=conversation.id,
        input=user_message,
        extra_body={"agent_reference": AGENT_REFERENCE},
    )

    while True:
        # Drain this stream, yielding to Gradio along the way
        for snapshot in _stream_response(stream, chat, in_progress_tools):
            yield snapshot

        pending = getattr(stream, "_pending_calls", []) or []
        last_id = getattr(stream, "_last_response_id", None)
        if not pending or last_id is None:
            return

        # Execute every pending function call and feed results back
        outputs: ResponseInputParam = []
        for item in pending:
            fn = function_dispatch.get(item.name)
            if fn is None:
                output = json.dumps({"error": f"unknown function {item.name}"})
            else:
                args = json.loads(item.arguments) if item.arguments else {}
                if DEBUG:
                    print(f"{item.name} inputs > {args} (call_id:{item.call_id})")
                output = fn(**args)
                if DEBUG:
                    print(f"output > {output}")
            outputs.append(FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=output,
            ))

        if DEBUG:
            print("status > resuming after function calls")
        stream = openai_client.responses.create(
            stream=True,
            input=outputs,
            previous_response_id=last_id,
            extra_body={"agent_reference": AGENT_REFERENCE},
        )

### Build a Gradio UI
Create a Gradio interface for interacting with the enterprise agent. 
Include a chatbot component and a text input box for user queries.

In [11]:
brand_theme = gr.themes.Default(
    primary_hue="blue",
    secondary_hue="blue",
    neutral_hue="gray",
    font=["Segoe UI", "Arial", "sans-serif"],
    font_mono=["Courier New", "monospace"],
    text_size="lg",
).set(
    button_primary_background_fill="#0f6cbd",
    button_primary_background_fill_hover="#115ea3",
    button_primary_background_fill_hover_dark="#4f52b2",
    button_primary_background_fill_dark="#5b5fc7",
    button_primary_text_color="#ffffff",
    button_secondary_background_fill="#e0e0e0",
    button_secondary_background_fill_hover="#c0c0c0",
    button_secondary_background_fill_hover_dark="#a0a0a0",
    button_secondary_text_color="#000000",
    body_background_fill="#f5f5f5",
    block_background_fill="#ffffff",
    body_text_color="#242424",
    body_text_color_subdued="#616161",
    block_border_color="#d1d1d1",
    block_border_color_dark="#333333",
    input_background_fill="#ffffff",
    input_border_color="#d1d1d1",
    input_border_color_focus="#0f6cbd",
)

# Shut down previous server from this notebook (if any) before building new UI
GRADIO_PORT = 7861
if 'demo' in globals():
    try:
        demo.close()
        print(f"closed previous Gradio server on port {GRADIO_PORT}")
    except Exception:
        pass

with gr.Blocks(theme=brand_theme, css="footer {visibility: hidden;}", fill_height=True) as demo:

    def clear_conversation():
        global conversation
        conversation = openai_client.conversations.create()
        return []

    def on_example_clicked(evt: gr.SelectData):
        return evt.value["text"]  # Fill the textbox with that example text

    gr.HTML("<h1 style=\"text-align: center;\">Azure AI Agent Service</h1>")
    
    chatbot = gr.Chatbot(
        examples=[
            {"text": "Join both CSV datasets for patient PATIENT_ID 10, run LOF outlier detection, generate a 2D scatter plot highlighting the patient against the cohort, and summarize the full clinical profile."},
            {"text": "Based on the analysis above, visualize patient PATIENT_ID 10 against the cohort with a scatter plot, search best practices for this patient's cancer stage and risk factors, then provide a personalized treatment recommendation with specific drug regimens. If no prior analysis exists, first analyze patient PATIENT_ID 10."},
            {"text": "Analyze patient PATIENT_ID 10 as an outlier using LOF, generate a scatter plot highlighting the patient against the cohort, and then recommend a stage-specific treatment plan based on best practices."},
        ],
        show_label=False,
        scale=1,
    )

    textbox = gr.Textbox(
        show_label=False,
        lines=1,
        submit_btn=True,
    )

    # Populate textbox when an example is clicked
    chatbot.example_select(fn=on_example_clicked, inputs=None, outputs=textbox)

    # On submit: call azure_enterprise_chat, then clear the textbox
    (textbox
     .submit(
         fn=azure_enterprise_chat,
         inputs=[textbox, chatbot],
         outputs=[chatbot, textbox],
     )
     .then(
         fn=lambda: "",
         outputs=textbox,
     )
    )

    # A "Clear" button that resets the conversation and the Chatbot
    chatbot.clear(fn=clear_conversation, outputs=chatbot)

# Launch your Gradio app
if __name__ == "__main__":
    print("Note:\ntool calling may fail, close the chat with trash bin icon (🗑️) and rerun the prompt.\n")
    demo.launch(server_port=GRADIO_PORT)

/var/folders/x6/_vk8l9r96m75w07cmf8y05n80000gn/T/ipykernel_57682/3296968647.py:38: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=brand_theme, css="footer {visibility: hidden;}", fill_height=True) as demo:


Note:
tool calling may fail, close the chat with trash bin icon (🗑️) and rerun the prompt.

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### (Optional) Delete agent version, conversation, and vector store resources
Uncomment the next cell to delete the resources created in this notebook.

In Projects v2, agents are versioned: `delete_version` removes a single version,
`delete` removes the agent and all its versions.

For custom code interpreter infrastructure, tear down via:
```bash
az group delete --name <YOUR_RESOURCE_GROUP> --yes --no-wait
```

In [12]:
# from azure.identity import DefaultAzureCredential
# from azure.ai.projects import AIProjectClient
# import os

# credential = DefaultAzureCredential()
# project_client_delete = AIProjectClient(
#    credential=credential,
#    endpoint=os.environ.get("PROJECT_ENDPOINT")
# )
# openai_client_delete = project_client_delete.get_openai_client()

# try:
#    project_client_delete.agents.delete_version(
#        agent_name=agent.name,
#        agent_version=agent.version,
#    )
#    print("Agent version deletion successful.")
#    openai_client_delete.conversations.delete(conversation.id)
#    print("Conversation deletion successful.")
#    openai_client_delete.vector_stores.delete(vector_store_id)
#    print("Vector store deletion successful.")
#    print("All deletions succeeded.")
# except Exception as e:
#    print(f"Error during deletion: {e}")

In [13]:
# existing_stores = openai_client.vector_stores.list()
# for store in existing_stores:
#     openai_client.vector_stores.delete(store.id)


user > Join both CSV datasets for patient PATIENT_ID 10, run LOF outlier detection, generate a 2D scatter plot highlighting the patient against the cohort, and summarize the full clinical profile.
status > created (response_id: resp_4c6746ebddc059fd0069f92c8e55048190b4423b1bdde0a515)
output_item > code_interpreter_call (id: ci_4c6746ebddc059fd0069f92c900e308190929ce1ff931bb1d2)
tool call > code_interpreter
tool done > code_interpreter
  captured container_id > cntr_69f92c8ed53881909f6fe1aeae5c463f0485348a0c7ad1d8
output_item > message (id: msg_4c6746ebddc059fd0069f92c9514d08190b684f0c5ddc431a6)
message done > (id: msg_4c6746ebddc059fd0069f92c9514d08190b684f0c5ddc431a6)
  content block > type=output_text
  === RAW BLOCK TEXT (last 500 chars) ===
  I have successfully joined the two datasets on PATIENT_ID and extracted the full profile of patient 10. 

Next, I will:
- Normalize the numeric features of the dataset,
- Run LOF (Local Outlier Factor) outlier detection,
- Identify the 2 most